<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/wnut17-robust-ner/notebooks/wnut17_multi_model_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -q seqeval
! pip install emoji==0.6.0

In [ ]:
import sys
import numpy as np
import torch, time
from torch import nn
import transformers
from transformers import (AutoTokenizer,
                          AutoModelForTokenClassification,
                          DataCollatorForTokenClassification,
                          TrainingArguments,
                          Trainer,
                          set_seed)

import datasets
from seqeval.metrics import classification_report, accuracy_score
from seqeval.scheme import IOB2

In [ ]:
wnut = datasets.load_dataset("flaitenberger/wnut_17")

In [ ]:
label_list = [label for label in wnut["train"].features["ner_tags"].feature.names]
id2label =   {i:label for i, label in enumerate(label_list)}
label2id =   {label:i for i, label in id2label.items()}


In [ ]:
shown = 0
for ex in wnut["train"]:

  if any(t != 0 for t in ex["ner_tags"]):
    print(" ".join(ex["tokens"]))
    print([(tok, id2label[t]) for tok, t in zip(ex["tokens"], ex["ner_tags"])])
    print()
  shown += 1

  if shown == 5:
    break

In [ ]:
for split in wnut:
  lens = [len(ex["tokens"]) for ex in wnut[split]]
  print(split, "mean:",round(np.mean(lens),2), "max: ", np.max(lens), "median: ", np.median(lens), "95th pct: ", np.quantile(lens, 0.95))

In [ ]:
from collections import Counter

for split in wnut:
  c = Counter(id2label[t] for ex in wnut[split] for t in ex["ner_tags"])
  ents = {l[2:]: n for l , n in c.items() if l.startswith("B-")}
  total = sum(c.values())
  print(split, "tokens:", total, "O share:", round(c["O"] / total, 3))
  print(ents, "total entities:", sum(ents.values()))

In [ ]:
tok = AutoTokenizer.from_pretrained("bert-base-cased")

words = ["Pxleyes", "Top", "50", "Photography", "Contest", "#photography"]

enc = tok(words, is_split_into_words= True)

print(tok.convert_ids_to_tokens(enc["input_ids"]))
print(enc.word_ids())

In [ ]:
print(enc.word_ids())
print(wnut["train"]["ner_tags"])

In [ ]:
def tokenize_and_align(batch):

  enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=256)
  all_labels = []
  for i, tags in enumerate(batch["ner_tags"]):
    prev = None
    labs = []
    for w in enc.word_ids(batch_index=i):
      if w is None:
        labs.append(-100)
      elif w != prev:
        labs.append(tags[w])
      else:
        labs.append(-100)
      prev = w

    all_labels.append(labs)
  enc["labels"] = all_labels

  return enc

In [ ]:
tokenized = wnut.map(tokenize_and_align, batched = True, remove_columns = wnut["train"].column_names)

for split in tokenized:
  lost = sum(len(ex["tokens"]) - sum(l != -100 for l in lab) for ex, lab in zip(wnut[split], tokenized[split]["labels"]))
  print(split, "max_tokens:", max(len(x) for x in tokenized[split]["input_ids"]), "words lost:", lost)

In [ ]:
val_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["validation"]]
test_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["test"]]

In [ ]:
model = AutoModelForTokenClassification.from_pretrained("bert-base-cased", num_labels=len(label_list), id2label=id2label, label2id=label2id)
print(model.classifier)
print(sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad))

collator = DataCollatorForTokenClassification(tok)
batch = collator([tokenized["train"][i] for i in range(3)])
print({k: v.shape for k, v in batch.items()})
print(batch["labels"][0])

In [ ]:
def evaluate_ner(gold, pred):

    rep = classification_report(
        gold,
        pred,
        mode="strict",
        scheme=IOB2,
        output_dict=True,
        zero_division=0)

    out = {"precision": rep["micro avg"]["precision"],
           "recall": rep["micro avg"]["recall"],
           "f1": rep["micro avg"]["f1-score"],
           "token_acc": accuracy_score(gold, pred)}

    out["per_type_f1"] = {k: v["f1-score"] for k, v in rep.items() if "avg" not in k}

    return out

In [ ]:
def to_tags(logits, labels):

  preds = np.argmax(logits, axis = -1)
  gold, pred = [], []
  for pr, lb in zip(preds, labels):
    gold.append([id2label[l] for l in lb if l != -100])
    pred.append([id2label[q] for q, l in zip(pr, lb) if l != -100])

  return gold, pred

def compute_metrics(p):
  gold, pred = to_tags(*p)
  r = evaluate_ner(gold, pred)
  return {
      "precision":r["precision"],
      "f1":r["f1"],
      "recall": r["recall"],
      "accuracy": r["token_acc"]}




In [ ]:
b = collator([tokenized["validation"][i] for i in range(64)])
labels = b["labels"].numpy()
logits = np.eye(len(label_list))[np.where(labels == -100, 0, labels)]
print(compute_metrics((logits, labels)))

In [ ]:
import torch, time
from transformers import Trainer, TrainingArguments, set_seed

set_seed(42)

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels = len(label_list),
    id2label = id2label,
    label2id = label2id)

args = TrainingArguments(
    output_dir="bert_base_cased_ckpt",
    learning_rate = 5e-5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 64,
    num_train_epochs = 10,
    weight_decay = 0.01,
    warmup_steps = 107,
    fp16  =True,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 1,
    load_best_model_at_end = True,
    metric_for_best_model = "f1",
    seed = 42,
    report_to = "none")


trainer = Trainer(model = model,
                  args = args,
                  train_dataset = tokenized["train"],
                  eval_dataset  = tokenized["validation"],
                  data_collator = collator,
                  processing_class = tok,
                  compute_metrics = compute_metrics)

# torch.cuda.reset_peak_memory_stats()
# t0 = time.time()

# trainer.train()


# train_time = time.time() - t0
# peak_gb = torch.cuda.max_memory_allocated() / 1e9
# print("train time (s):", round(train_time), "peak GPU memory (GB):", round(peak_gb, 2))

In [ ]:
out = trainer.predict(tokenized["validation"])
gold, pred = to_tags(out.predictions, out.label_ids)
print(evaluate_ner(gold, pred))
print(gold == val_gold)
print(trainer.state.best_metric, trainer.state.best_model_checkpoint)

#**Architecture: RoBERTa-base**

In [ ]:
MODEL_NAME = "FacebookAI/roberta-base"
tok_rb = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space = True)
enc = tok_rb(words, is_split_into_words= True)

In [ ]:
def tokenize_and_align_rb(batch):
  enc = tok_rb(batch["tokens"],
               is_split_into_words= True,
               truncation= True,
               max_length = 256)

  all_labels = []
  for i, tags in enumerate(batch["ner_tags"]):

    prev = None
    labs = []
    for w in enc.word_ids(batch_index = i):

      if w == None:
        labs.append(-100)
      elif w != prev:
        labs.append(tags[w])
      else:
        labs.append(-100)
      prev = w

    all_labels.append(labs)

  enc["labels"] = all_labels

  return enc


In [ ]:
tokenized_rb = wnut.map(tokenize_and_align_rb, batched = True, remove_columns = wnut["train"].column_names)

for split in tokenized_rb:
  lost = sum(len(ex["tokens"]) - sum(l != -100 for l in lab) for ex, lab in zip(wnut[split], tokenized_rb[split]["labels"]))
  print(split, "max tokens", max(len(x) for x in tokenized_rb[split]["input_ids"]), "words lost", lost)

In [ ]:
val_gold =  [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["validation"]]
test_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["test"]]

In [ ]:
MODEL_NAME = "FacebookAI/roberta-base"

model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME,
                                                        num_labels = len(label_list),
                                                        id2label= id2label,
                                                        label2id = label2id)

In [ ]:
print(model.classifier)
print()
print("parameters without gradient",sum(p.numel() for p in model.parameters()))
print()
print("Parameters with gradient",sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
collator = DataCollatorForTokenClassification(tok_rb)

In [ ]:
import torch, time
from transformers import Trainer, TrainingArguments, set_seed, EarlyStoppingCallback

args = TrainingArguments(
    output_dir = "roberta_base_ckpt",
    num_train_epochs = 20,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 64,
    learning_rate = 5e-5,
    warmup_steps = 107,
    weight_decay = 0.01,
    save_strategy= "epoch",
    eval_strategy = "epoch",
    seed= 42,
    metric_for_best_model = "f1",
    report_to = "none",
    load_best_model_at_end = True,
    save_total_limit = 1,
    fp16 = True,
    greater_is_better = True
)

trainer = Trainer(
    model = model,
    args = args,
    data_collator = collator,
    train_dataset= tokenized_rb["train"],
    eval_dataset= tokenized_rb["validation"],
    processing_class= tok_rb,
    compute_metrics = compute_metrics,
    callbacks = [
        EarlyStoppingCallback(
        early_stopping_patience = 2,
        early_stopping_threshold = 0.001
        )
    ])

torch.cuda.reset_peak_memory_stats()

t0 = time.time()

trainer.train()

train_time = time.time() - t0

peak_gb = torch.cuda.max_memory_allocated() / 1e9

print("train time (s)", train_time, "peak GPU memory", peak_gb)

In [ ]:
out = trainer.predict(tokenized_rb["validation"])
gold, pred = to_tags(out.predictions, out.label_ids)
print(evaluate_ner(gold, pred))
print(gold == val_gold)
print(trainer.state.best_metric, trainer.state.best_model_checkpoint)

# **Distilbert Base Cased**

In [ ]:
MODEL_NAME = "distilbert-base-cased"

In [ ]:
tok_dis = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def tokenize_and_align_dis(batch):
  enc = tok_dis(batch["tokens"], max_length = 256, truncation = True, is_split_into_words= True)
  all_labels = []

  for i, tags in enumerate(batch["ner_tags"]):
    labs = []
    prev = None
    for w in enc.word_ids(batch_index=i):

      if w == None:
        labs.append(-100)
      elif w != prev:
        labs.append(tags[w])
      else:
        labs.append(-100)
      prev = w
    all_labels.append(labs)

  enc["labels"] = all_labels

  return enc


In [ ]:
tokenize_dis = wnut.map(tokenize_and_align_dis, batched = True, remove_columns = wnut["train"].column_names)

for split in tokenize_dis:
  lost = sum(len(ex["tokens"]) - sum(l != -100 for l in lab) for ex, lab in zip(wnut[split], tokenize_dis[split]["labels"]))
  print(split, "max tokens", max(len(x) for x in tokenize_dis[split]["input_ids"]), "words lost", lost)

In [ ]:
model_dis = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels = len(label_list),
    id2label=id2label,
    label2id=label2id)

In [ ]:
collator_dis = DataCollatorForTokenClassification(tok_dis)

In [ ]:
args_dis = TrainingArguments(
    output_dir= "distilbert-base-cased_ckpt",
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 64,
    learning_rate = 5e-5,
    num_train_epochs = 10,
    warmup_steps = 107,
    weight_decay = 0.01,
    fp16 = True,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 1,
    metric_for_best_model = "f1",
    greater_is_better = True,
    report_to = "none",
    load_best_model_at_end = True

)

trainer_dis = Trainer(

    model = model_dis,
    args=  args_dis,
    data_collator = collator_dis,
    train_dataset = tokenize_dis["train"],
    eval_dataset = tokenize_dis["validation"],
    compute_metrics = compute_metrics,
    processing_class = tok_dis,
    callbacks = [EarlyStoppingCallback(
        early_stopping_patience = 5,
        early_stopping_threshold = 0.0001
    )],

)

torch.cuda.reset_peak_memory_stats()

t0 = time.time()

trainer_dis.train()

train_time = time.time() - t0

peak_gb = torch.cuda.max_memory_allocated() / 1e9

print("Training time", train_time, "peak GPU memory", peak_gb)

In [ ]:
out = trainer_dis.predict(tokenize_dis["validation"])
gold, pred = to_tags(out.predictions, out.label_ids)
print(evaluate_ner(gold, pred))
print(gold == val_gold)
print(trainer_dis.state.best_metric, trainer_dis.state.best_model_checkpoint)

# **Vinai Bertweet**

In [85]:
MODEL_NAME = "vinai/bertweet-base"
tok_vin = AutoTokenizer.from_pretrained(MODEL_NAME)

In [86]:
def tokenize_and_align_vin(batch):
  ids, labels = [], []
  for toks, tags in zip(batch["tokens"], batch["ner_tags"]):
    x, y = [tok_vin.cls_token_id], [-100]
    for w, t in zip(toks, tags):
      p = tok_vin.encode(w, add_special_tokens = False)
      x += p
      y += [t] + [-100] * (len(p) - 1)
    x, y = x[:127] + [tok_vin.sep_token_id], y[:127] + [-100]
    ids.append(x); labels.append(y)
  return {"input_ids": ids, "attention_mask": [[1] * len(i) for i in ids], "labels": labels}

In [87]:
vin_tokenized = wnut.map(tokenize_and_align_vin, batched = True, remove_columns= wnut["train"].column_names)

In [88]:
for split in vin_tokenized:
  lost = sum(len(ex["tokens"]) - sum(l != -100 for l in lab) for ex, lab in zip(wnut[split], vin_tokenized[split]["labels"]))
  print(split, "max tokens", max(len(x) for x in vin_tokenized[split]["input_ids"]), "words lost", lost)


train max tokens 68 words lost 0
validation max tokens 84 words lost 0
test max tokens 128 words lost 13


In [93]:
vin_model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, id2label= id2label, label2id=label2id, num_labels = len(label_list))


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [94]:
vin_collator = DataCollatorForTokenClassification(tok_vin)

In [95]:
vin_args = TrainingArguments(
    output_dir = "bertweet_base_ckpt",
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 64,
    save_strategy = "epoch",
    eval_strategy = "epoch",
    num_train_epochs = 10,
    learning_rate = 5e-5,
    fp16 = True,
    report_to = "none",
    weight_decay = 0.01,
    load_best_model_at_end = True,
    warmup_steps = 107,
    save_total_limit = 1,
    metric_for_best_model = "f1",
    seed = 42,
    greater_is_better = True

)

In [96]:
vin_trainer = Trainer(
    model = vin_model,
    args = vin_args,
    data_collator = vin_collator,
    compute_metrics = compute_metrics,
    train_dataset = vin_tokenized["train"],
    eval_dataset = vin_tokenized["validation"],
    processing_class = tok_vin,

)

torch.cuda.reset_peak_memory_stats()

t0 = time.time()
vin_trainer.train()
train_time = time.time() - t0

peak_gb = torch.cuda.max_memory_allocated() / 1e9

print("Training time", train_time, "peak GPU memory", peak_gb)

Epoch,Training Loss,Validation Loss,Precision,F1,Recall,Accuracy
1,No log,0.341422,0.000000,0.000000,0.000000,0.920549
2,No log,0.215036,0.662771,0.553310,0.474880,0.949215
3,No log,0.168598,0.689986,0.642812,0.601675,0.961228
4,No log,0.174975,0.736544,0.674449,0.622010,0.964597
5,0.252396,0.198378,0.746291,0.666225,0.601675,0.962753
6,0.252396,0.200414,0.732541,0.653413,0.589713,0.962054
7,0.252396,0.196816,0.722772,0.662346,0.611244,0.962499
8,0.252396,0.194519,0.735127,0.673152,0.620813,0.964025
9,0.252396,0.201414,0.731092,0.673548,0.624402,0.963770
10,0.022481,0.200009,0.719178,0.670498,0.627990,0.963834


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training time 337.11215472221375 peak GPU memory 5.046646784
